# QA de red multimodal — Valencia

Genera y valida las redes OSM (driving/walking/bike) y GTFS (train/bus) en tres formatos de salida (GeoJSON, Shapefile, GeoPackage), usando los checks de `network_qa.py`.


## 1. Configuración

In [5]:
from pathlib import Path

pkg_root = Path.home() / "Desktop/Proyectos/2026/02_Libreria_multimodal/multimodalpy"
print("¿existe la ruta?", pkg_root.exists())
print("contenido:")
for p in sorted(pkg_root.rglob("*.py")):
    print(" ", p.relative_to(pkg_root))

¿existe la ruta? True
contenido:
  __init__.py
  get_area.py
  get_network.py
  network_qa.py
  process_gtfs.py
  process_osm.py


In [7]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

sys.path.insert(0, str(Path.home() / "Desktop/Proyectos/2026/02_Libreria_multimodal"))
from multimodalpy.get_network import main

# --- Rutas y credenciales -------------------------------------------------
# La API key NO debe vivir en texto plano en el notebook: se lee de una
# variable de entorno. Antes de correr esto, exporta:
#   export NAP_API_KEY="tu-clave-aqui"
# (o defínela en un archivo .env que cargues con python-dotenv).
NAP_API_KEY = os.environ["NAP_API_KEY"]

output_folder = Path.home() / "Desktop/descargas_multimodalpy/output_Valencia"
boundaries = Path.home() / "Desktop/descargas_multimodalpy/input_shapes/capa_Valencia.shp"

AREA_NAME = "Valencia"
CRS = "epsg:25830"
MODES = ["driving", "walking", "bike", "bus", "train"]


ModuleNotFoundError: No module named 'multimodalpy.get_network'

## 2. Funciones de validación

El código de los checks vive en `network_qa.py`, en la misma carpeta que este notebook (antes estaba pegado en una celda de 26.000 caracteres — mantenerlo como módulo aparte permite editarlo, testearlo y reusarlo desde otros notebooks o scripts sin duplicar código).

In [ ]:
from network_qa import (
    run_all_checks_osm,
    results_to_frame,
    style_results,
    summary_matrix,
    style_summary_matrix,
    show_issues,
)


## 3. Carga de nodos/aristas por modo y formato

`STEM_BY_MODE` mapea cada modo al prefijo de archivo que genera `main()`. `load_network()` centraliza cómo se lee cada formato, así que nodos y aristas siempre se asignan a variables distintas — la razón original del bug en la sección de GeoPackage era reutilizar el mismo nombre de variable para las dos capas.

`run_checks_for_all_modes()` corre con `verbose=False`: en vez de imprimir ~27 líneas por modo (135 líneas en total, difíciles de escanear), guarda los resultados de cada modo en un diccionario para revisarlos como tabla en la siguiente sección.

In [ ]:
# Prefijo de archivo que genera main() para cada modo. Los dos GTFS llevan
# fecha/hora de descarga en el nombre: si cambian en cada ejecución, resuelve
# el prefijo real con glob en vez de hardcodearlo aquí.
STEM_BY_MODE = {
    "driving": "osm_driving",
    "walking": "osm_walking",
    "bike": "osm_bike",
    "train": "gtfs_20260902_080005_metro_valencia",
    "bus": "gtfs_20260908_020020_genvalenciana_interurbano",
}


def load_network(mode: str, fmt: str, gpkg_path=None):
    """Devuelve (nodes, edges) para `mode` en el formato `fmt`.

    fmt: "geojson" | "shapefile" | "geopackage"

    `output_folder`/`gpkg_path` se convierten a Path aqui mismo, para que
    esta funcion funcione sin importar si en la celda de configuracion los
    definiste como str o como Path (el operador "/" solo existe entre
    objetos Path, y ese fue el TypeError si veniamos de un str).
    """
    stem = STEM_BY_MODE[mode]
    folder = Path(output_folder)

    if fmt == "geojson":
        nodes = gpd.read_file(folder / f"{stem}_nodes.geojson")
        edges = gpd.read_file(folder / f"{stem}_edges.geojson")
    elif fmt == "shapefile":
        nodes = gpd.read_file(folder / f"{stem}_nodes.shp")
        edges = gpd.read_file(folder / f"{stem}_edges.shp")
    elif fmt == "geopackage":
        if gpkg_path is None:
            raise ValueError("fmt='geopackage' requiere gpkg_path=...")
        gpkg_path = Path(gpkg_path)
        nodes = gpd.read_file(gpkg_path, layer=f"{stem}_nodes")
        edges = gpd.read_file(gpkg_path, layer=f"{stem}_edges")
    else:
        raise ValueError(f"Formato desconocido: {fmt!r}")

    return nodes, edges


def run_checks_for_all_modes(fmt: str, gpkg_path=None) -> dict:
    """Corre run_all_checks_osm para cada modo en MODES, en el formato dado.

    Devuelve {modo: [CheckResult, ...]} — usa `show_results()` (celda de
    abajo) para verlo como tabla en vez de leer el diccionario a mano.
    """
    results_by_mode = {}
    for mode in MODES:
        nodes, edges = load_network(mode, fmt, gpkg_path=gpkg_path)
        # verbose=False: nada de log linea-por-linea aqui, solo guardamos
        # los resultados. El resumen se revisa en la siguiente celda como
        # tabla coloreada, mucho mas facil de escanear que el print crudo.
        results_by_mode[mode] = run_all_checks_osm(nodes, edges, verbose=False)
    return results_by_mode


## 4. Cómo leer los resultados

En vez de scrollear un log de texto, dos vistas:

1. **Matriz check × modo**, coloreada — verde = 0 problemas, rojo = hay problemas, gris = el check no corrió para ese modo (p. ej. columna ausente). De un vistazo se ve qué modo falla y en qué check.
2. **`show_issues(resultados, "nombre del check", n)`** — abre el detalle (las primeras `n` filas problemáticas) de un check concreto, solo cuando hace falta mirarlo de cerca.


In [ ]:
def show_results(results_by_mode: dict):
    """Muestra la matriz coloreada check x modo para un formato."""
    matrix = summary_matrix(results_by_mode)
    return style_summary_matrix(matrix)


## 5. GeoJSON

In [ ]:
main(
    area_name=AREA_NAME,
    modes=MODES,
    output_path=str(output_folder),
    crs=CRS,
    output_file_type="geojson",
    boundaries_path=str(boundaries),
    api_key=NAP_API_KEY,
)

results_geojson = run_checks_for_all_modes("geojson")
show_results(results_geojson)


Ejemplo de cómo abrir el detalle de un check que salió en rojo arriba (ajusta el nombre del check y el modo a lo que veas en tu matriz):

```python
show_issues(results_geojson["bus"], "trip_count == sum(hourly_trip_counts)")
```

## 6. Shapefile

In [ ]:
main(
    area_name=AREA_NAME,
    modes=MODES,
    output_path=str(output_folder),
    crs=CRS,
    output_file_type="shapefile",
    boundaries_path=str(boundaries),
    api_key=NAP_API_KEY,
)

results_shapefile = run_checks_for_all_modes("shapefile")
show_results(results_shapefile)


## 7. GeoPackage

⚠️ ESRI Shapefile trunca nombres de columna a 10 caracteres (`from_node_id` → `from_node_`), lo cual ya se refleja en la matriz de la sección anterior (columnas "ausentes" en gris/rojo que aquí deberían verse en verde). GeoPackage no tiene ese límite — compara ambas matrices para confirmar que el dato en sí está bien y el problema es solo del formato Shapefile.

In [ ]:
main(
    area_name=AREA_NAME,
    modes=MODES,
    output_path=str(output_folder),
    crs=CRS,
    output_file_type="geopackage",
    boundaries_path=str(boundaries),
    api_key=NAP_API_KEY,
)

gpkg_path = output_folder / "valencia.gpkg"
results_geopackage = run_checks_for_all_modes("geopackage", gpkg_path=gpkg_path)
show_results(results_geopackage)


## Notas de esta revisión

- **Bug corregido**: en el notebook original, la sección de GeoPackage asignaba `df_<modo>_nodes` dos veces (una por capa) y nunca creaba `df_<modo>_edges`, así que los checks de esa sección corrían sobre datos de la sección anterior (Shapefile). `load_network()` ahora devuelve ambos objetos en una sola llamada, así que no hay nombre que pueda pisarse.
- **Legibilidad de resultados**: `run_all_checks_osm(..., verbose=False)` + `summary_matrix()`/`style_summary_matrix()` reemplazan el log de texto línea-por-línea por una matriz check × modo coloreada; `show_issues()` abre el detalle de un check puntual solo cuando hace falta.
- **Seguridad**: la API key se lee de `NAP_API_KEY` como variable de entorno en vez de estar escrita en el notebook. Si la clave que tenías antes en texto plano se llegó a compartir o subir a un repo, te recomendaría regenerarla.
- **Portabilidad**: las rutas usan `Path.home()` en vez de un usuario hardcodeado (`/Users/aidavillalba/...`), para que el notebook corra en cualquier máquina sin editar rutas a mano.
- Si `STEM_BY_MODE` cambia en cada ejecución (por la fecha/hora en los nombres GTFS), conviene resolverlo con `glob` en vez de escribir el prefijo a mano cada vez.
